# Análise — Case Medipreço "O fechamento do mês"

Consultas que sustentam os achados da proposta (`docs/PROPOSTA.md`). Tudo em SQL sobre `dados/base.sqlite`, para dar para reler e refazer.

Duas linhas do tempo aparecem em todas as consultas:
- **visão do sistema**: o que uma consulta ao sistema mostraria num instante (usa `data_processamento`);
- **data informada**: o que de fato aconteceu, segundo o RH (usa `data_informada`).

**Linha do tempo adotada (Mateus, 17/11): a base cadastrada.** Vale o que estava cadastrado no sistema. A posição de um dia é a do fim daquele dia. A data informada aparece só para comparação.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 200, "display.width", 200)
DB = next(p for p in [Path("dados/base.sqlite"), Path("../dados/base.sqlite")] if p.exists())
con = sqlite3.connect(DB)

def q(sql, **params):
    """Roda uma consulta e devolve um DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

## 0. Tabelas de apoio

- `ev`: eventos com o cliente e o atraso em dias entre o fato e o processamento.
- `intervalos`: um período de vigência por pessoa e CNPJ (entrada casada com a saída seguinte), nas duas linhas do tempo. `anulado = 1` marca o que o suporte desfez.
- `dias`: calendário de 01/06 a 12/11/2026.
- `foto`: a visão do sistema em cada instante `contagem_em` das notas.

In [2]:
con.executescript('''
drop table if exists ev; drop table if exists intervalos; drop table if exists dias; drop table if exists foto;

create temp table ev as
select v.*, e.cliente, e.razao_social,
       date(v.data_processamento) as dia_proc,
       cast(julianday(date(v.data_processamento)) - julianday(v.data_informada) as integer) as atraso_dias
from eventos v join empresas e using (cnpj);

create temp table intervalos as
with es as (
  select *,
         lead(tipo_evento)        over w as prox_tipo,
         lead(data_informada)     over w as prox_data,
         lead(data_processamento) over w as prox_proc,
         lead(origem)             over w as prox_origem
  from ev where tipo_evento in ('entrada', 'saida')
  window w as (partition by pessoa_id, cnpj order by data_processamento, evento_id)
)
select es.pessoa_id, es.cnpj, es.cliente, es.razao_social, p.tipo,
       es.data_informada      as inicio,       -- data informada
       es.prox_data           as fim,
       es.data_processamento  as proc_inicio,  -- visão do sistema
       es.prox_proc           as proc_fim,
       coalesce(es.prox_origem = 'suporte_medipreco', 0) as anulado
from es join pessoas p using (pessoa_id)
where es.tipo_evento = 'entrada';

create temp table dias as
with recursive d(dia) as (select '2026-06-01' union all select date(dia, '+1 day') from d where dia < '2026-11-12')
select dia, substr(dia, 1, 7) as mes from d;

create temp table foto as
select t.contagem_em, x.cliente, x.cnpj, x.pessoa_id, x.plano, p.tipo
from (select distinct contagem_em from notas_emitidas) t
join (select ev.*, n.contagem_em as t_ref,
             row_number() over (partition by n.contagem_em, ev.pessoa_id, ev.cnpj
                                order by ev.data_processamento desc, ev.evento_id desc) as rn
      from ev join (select distinct contagem_em from notas_emitidas) n
        on ev.data_processamento <= n.contagem_em) x
  on x.t_ref = t.contagem_em and x.rn = 1 and x.tipo_evento <> 'saida'
join pessoas p using (pessoa_id);
''')
q("select (select count(*) from ev) eventos, (select count(*) from intervalos) intervalos, (select count(*) from dias) dias, (select count(*) from foto) linhas_foto")

,eventos,intervalos,dias,linhas_foto
0,13085,10417,165,32272


## 1. Como as notas foram feitas: nota × visão do sistema

Para cada item de nota, o que a visão do sistema mostrava em `contagem_em`. Se bate, a nota foi feita por foto.

In [3]:
q('''
select n.cliente, n.competencia, n.item, n.base_calculo, n.contagem_em,
       n.quantidade as cobrado,
       sum(f.tipo = 'titular') as foto_titulares,
       count(f.pessoa_id)      as foto_vidas,
       case when n.quantidade = case when n.base_calculo = 'titulares' then sum(f.tipo = 'titular') else count(f.pessoa_id) end
            then 'bate' else 'NÃO BATE' end as resultado
from notas_emitidas n
left join foto f
  on f.contagem_em = n.contagem_em
 and f.cliente = n.cliente
 and (n.cliente <> 'Grupo Cedro' or f.cnpj = n.cnpj_tomador)
 and (n.item not like '%Plano%' or n.item like '%' || f.plano)
group by n.nota_id, n.item
order by resultado desc, n.cliente, n.competencia
''')

,cliente,competencia,item,base_calculo,contagem_em,cobrado,foto_titulares,foto_vidas,resultado
0,Aurora Logística,2026-08,Mensalidade Plano Base,vidas,2026-08-26 09:00:00,698,458,698,bate
1,Aurora Logística,2026-08,Mensalidade Plano Plus,vidas,2026-08-26 09:00:00,177,119,177,bate
2,Aurora Logística,2026-09,Mensalidade Plano Base,vidas,2026-09-25 09:00:00,701,458,701,bate
3,Aurora Logística,2026-09,Mensalidade Plano Plus,vidas,2026-09-25 09:00:00,177,119,177,bate
4,Aurora Logística,2026-10,Mensalidade Plano Base,vidas,2026-10-26 09:00:00,548,360,548,bate
5,Aurora Logística,2026-10,Mensalidade Plano Plus,vidas,2026-10-26 09:00:00,189,126,189,bate
6,Bravo Tecnologia,2026-08,Mensalidade,titulares,2026-07-24 09:00:00,321,321,483,bate
7,Bravo Tecnologia,2026-10,Mensalidade,titulares,2026-09-25 09:00:00,337,337,504,bate
8,Bravo Tecnologia,2026-11,Mensalidade,titulares,2026-10-26 09:00:00,347,347,518,bate
9,Delta Benefícios,2026-08,Mensalidade,vidas,2026-08-26 09:00:00,5051,4610,5051,bate


## 2. Portal × carga: como cada cliente informa

- `portal_rh`: o RH mexe pessoa a pessoa no portal. Um evento por vez, quase no mesmo dia do fato.
- `carga_rh`: o RH manda um arquivo. Vários eventos de uma vez, em poucos dias do mês.

In [4]:
q('''
select cliente, origem,
       count(*)                 as eventos,
       count(distinct dia_proc) as dias_com_movimento,
       round(1.0 * count(*) / count(distinct dia_proc), 1) as eventos_por_dia,
       round(avg(atraso_dias), 1) as atraso_medio_dias,
       max(atraso_dias)           as atraso_max_dias
from ev
where data_processamento >= '2026-06-01'
  and origem <> 'suporte_medipreco'
  and (pessoa_id, cnpj) not in (select pessoa_id, cnpj from intervalos where anulado = 1)
group by cliente, origem
order by atraso_medio_dias
''')

,cliente,origem,eventos,dias_com_movimento,eventos_por_dia,atraso_medio_dias,atraso_max_dias
0,Bravo Tecnologia,portal_rh,196,88,2.2,0.8,6
1,Faro Serviços,portal_rh,217,97,2.2,0.9,6
2,Aurora Logística,carga_rh,433,24,18.0,3.0,6
3,Grupo Cedro,carga_rh,595,25,23.8,3.6,7
4,Delta Benefícios,carga_rh,2134,6,355.7,15.2,32
5,Estrela Indústria,carga_rh,176,6,29.3,16.5,34


## 3. Estrela: o atraso é estrutural

O contrato (5.3) dá 5 dias úteis para informar. A Estrela manda **um arquivo por mês, por volta do dia 5**, com o movimento do mês anterior. O prazo é impossível de cumprir desse jeito, e a foto do dia 26 nunca vê o que aconteceu depois do dia 5.

In [5]:
q('''
select dia_proc as carga, tipo_evento, count(*) as eventos,
       min(data_informada) as fato_mais_antigo, max(data_informada) as fato_mais_recente,
       round(avg(atraso_dias), 1) as atraso_medio_dias,
       sum(atraso_dias > 7) as fora_de_5_dias_uteis_aprox
from ev
where cliente = 'Estrela Indústria' and data_processamento >= '2026-06-01'
group by dia_proc, tipo_evento
order by dia_proc, tipo_evento
''')

,carga,tipo_evento,eventos,fato_mais_antigo,fato_mais_recente,atraso_medio_dias,fora_de_5_dias_uteis_aprox
0,2026-06-05,entrada,2,2026-06-02,2026-06-02,3.0,0
1,2026-06-05,saida,5,2026-06-01,2026-06-02,3.8,0
2,2026-07-06,entrada,17,2026-06-11,2026-07-03,17.8,14
3,2026-07-06,saida,11,2026-06-18,2026-07-02,9.6,6
4,2026-08-05,entrada,11,2026-07-06,2026-07-30,23.0,10
5,2026-08-05,saida,18,2026-07-08,2026-08-04,14.6,10
6,2026-09-08,entrada,20,2026-08-13,2026-09-04,13.4,12
7,2026-09-08,saida,14,2026-08-05,2026-08-31,20.0,14
8,2026-10-05,entrada,13,2026-09-11,2026-09-25,16.2,13
9,2026-10-05,saida,12,2026-09-09,2026-10-01,13.9,9


In [6]:
# Outubro: o que a foto mostrou × quem estava ativo em 26/10 com o que se sabe hoje
q('''
select
  (select count(*) from foto where cliente = 'Estrela Indústria' and contagem_em = '2026-10-26 09:00:00') as foto_26_10,
  (select count(*) from intervalos where cliente = 'Estrela Indústria'
     and inicio <= '2026-10-26' and (fim is null or fim >= '2026-10-26'))                                 as ativos_26_10_data_informada,
  (select count(*) from intervalos where cliente = 'Estrela Indústria' and fim is null)                    as ativos_hoje,
  500 as minimo_contratual
''')

,foto_26_10,ativos_26_10_data_informada,ativos_hoje,minimo_contratual
0,510,479,475,500


## 4. Aurora: os três números de outubro (737 × 723 × 718)

- **737**: visão do sistema em 26/10 às 9h. Foi o cobrado.
- **723**: quem estava ativo em 26/10 pela data informada.
- **718**: visão do sistema em 31/10. É o que o RH baixou.

In [7]:
q('''
with estado as (  -- visão do sistema num instante
  select '26/10 09h (cobrado)' as visao, '2026-10-26 09:00:00' as t union all
  select '31/10 fim do dia (RH)',         '2026-10-31 23:59:59'
)
select e.visao, count(*) as vidas
from estado e join intervalos i
  on i.cliente = 'Aurora Logística' and i.proc_inicio <= e.t and (i.proc_fim is null or i.proc_fim > e.t)
group by e.visao
union all
select '26/10 pela data informada', count(*)
from intervalos where cliente = 'Aurora Logística' and inicio <= '2026-10-26' and (fim is null or fim >= '2026-10-26')
''')

,visao,vidas
0,26/10 09h (cobrado),737
1,31/10 fim do dia (RH),718
2,26/10 pela data informada,723


In [8]:
# As 19 pessoas da diferença: saídas processadas depois da foto e até 31/10
q('''
select pessoa_id, plano, data_informada as desligamento, data_processamento as entrou_no_sistema, atraso_dias,
       case when data_informada < '2026-10-26' then 'já tinha saído no dia 26'
            when data_informada = '2026-10-26' then 'saiu no dia 26'
            else 'saiu depois do dia 26' end as situacao
from ev
where cliente = 'Aurora Logística' and tipo_evento = 'saida'
  and data_processamento > '2026-10-26 09:00:00' and data_processamento <= '2026-10-31 23:59:59'
order by data_informada, pessoa_id
''')

,pessoa_id,plano,desligamento,entrou_no_sistema,atraso_dias,situacao
0,P430765,Plus,2026-10-21,2026-10-27 11:57:00,6,já tinha saído no dia 26
1,P636041,Plus,2026-10-21,2026-10-27 11:57:01,6,já tinha saído no dia 26
2,P895298,Plus,2026-10-21,2026-10-27 11:57:00,6,já tinha saído no dia 26
3,P993220,Base,2026-10-21,2026-10-27 11:57:00,6,já tinha saído no dia 26
4,P235668,Plus,2026-10-22,2026-10-27 11:57:01,5,já tinha saído no dia 26
5,P235985,Plus,2026-10-22,2026-10-27 11:57:02,5,já tinha saído no dia 26
6,P264468,Plus,2026-10-22,2026-10-27 11:57:02,5,já tinha saído no dia 26
7,P592239,Plus,2026-10-22,2026-10-27 11:57:01,5,já tinha saído no dia 26
8,P214492,Base,2026-10-23,2026-10-27 11:57:03,4,já tinha saído no dia 26
9,P259695,Plus,2026-10-23,2026-10-27 11:57:02,4,já tinha saído no dia 26


## 5. Bravo: por que a cláusula 4.4 nunca foi aplicada

O contrato pede o acerto entre o cobrado antecipadamente e os "titulares efetivamente ativos no mês". Nenhuma nota tem item de compensação. Para calcular o efetivo é preciso olhar o mês inteiro, e o processo de hoje só tem a foto. Abaixo, o que cada leitura de "efetivo" daria.

In [9]:
q('''
with por_dia as (
  select d.mes, d.dia, count(*) as titulares
  from dias d join intervalos i
    on i.cliente = 'Bravo Tecnologia' and i.tipo = 'titular'
   and i.proc_inicio <= d.dia || ' 23:59:59' and (i.proc_fim is null or i.proc_fim > d.dia || ' 23:59:59')
  group by d.mes, d.dia
),
distintos as (
  select d.mes, count(distinct i.pessoa_id) as titulares_distintos
  from dias d join intervalos i
    on i.cliente = 'Bravo Tecnologia' and i.tipo = 'titular'
   and i.proc_inicio <= d.dia || ' 23:59:59' and (i.proc_fim is null or i.proc_fim > d.dia || ' 23:59:59')
  group by d.mes
)
select n.competencia, n.quantidade as cobrado, n.contagem_em,
       min(p.titulares) as minimo, round(avg(p.titulares), 1) as media, max(p.titulares) as maximo,
       max(case when substr(p.dia, 9, 2) = '26' then p.titulares end) as no_dia_26,
       x.titulares_distintos as passaram_pelo_mes
from notas_emitidas n
join por_dia p on p.mes = n.competencia
join distintos x on x.mes = n.competencia
where n.cliente = 'Bravo Tecnologia'
group by n.competencia
''')

,competencia,cobrado,contagem_em,minimo,media,maximo,no_dia_26,passaram_pelo_mes
0,2026-08,321,2026-07-24 09:00:00,323,328.7,334,333.0,343
1,2026-09,499,2026-08-26 09:00:00,331,334.1,337,336.0,341
2,2026-10,337,2026-09-25 09:00:00,335,342.7,355,348.0,357
3,2026-11,347,2026-10-26 09:00:00,355,356.8,361,NaN,365


## 6. Delta: o lote anulado e a regra de "ao menos 1 dia"

Em 07/10 uma carga do RH incluiu 600 titulares; em 09/10 o suporte da Medipreço desfez as 600 entradas. Fatos:
- não eram pessoas repetidas: nenhuma tinha evento anterior, vínculo em outro CNPJ ou dependente;
- não foram cobradas: nenhuma foto das notas emitidas contém essas pessoas;
- pela letra do contrato (7.2 e 7.3) seriam cobráveis. A Medipreço já tratou o caso ao desfazer a carga; aqui fica só o registro. O motor precisa reconhecer a anulação para não cobrar por engano.

In [10]:
# O lote: 600 entradas em 3 minutos, desfeitas pelo suporte dois dias depois
q('''
select count(*) as pessoas,
       min(proc_inicio) as carga_inicio, max(proc_inicio) as carga_fim,
       min(inicio) as admissao_mais_antiga, max(inicio) as admissao_mais_recente,
       min(proc_fim) as anulacao_inicio, max(proc_fim) as anulacao_fim,
       min(fim) as data_informada_da_saida,
       (select count(*) from pessoas where titular_id in (select pessoa_id from intervalos where anulado = 1)) as dependentes,
       (select count(*) from ev where pessoa_id in (select pessoa_id from intervalos where anulado = 1)
          and data_processamento > '2026-10-09 23:59:59') as eventos_depois_da_anulacao
from intervalos where anulado = 1
''')

,pessoas,carga_inicio,carga_fim,admissao_mais_antiga,admissao_mais_recente,anulacao_inicio,anulacao_fim,data_informada_da_saida,dependentes,eventos_depois_da_anulacao
0,600,2026-10-07 10:31:00,2026-10-07 10:33:29,2024-01-01,2026-06-19,2026-10-09 16:05:00,2026-10-09 16:07:29,2026-10-07,0,0


In [11]:
# O lote já existia na base? Foi cobrado em alguma nota?
q('''
select
  (select count(*) from ev where pessoa_id in (select pessoa_id from intervalos where anulado = 1)
     and data_processamento < '2026-10-07')                                                    as eventos_antes_do_lote,
  (select count(*) from vinculos where pessoa_id in (select pessoa_id from intervalos where anulado = 1)
     and cnpj <> '60.184.533/0001-10')                                                         as vinculos_em_outro_cnpj,
  (select count(*) from foto where pessoa_id in (select pessoa_id from intervalos where anulado = 1)) as vezes_em_foto_de_nota,
  (select count(*) from vinculos where pessoa_id in (select pessoa_id from intervalos where anulado = 1)
     and ativo = 1)                                                                            as ativos_hoje
''')

,eventos_antes_do_lote,vinculos_em_outro_cnpj,vezes_em_foto_de_nota,ativos_hoje
0,0,0,0,0


In [12]:
# Vidas cadastradas em ao menos 1 dia do mês (base cadastrada), sem e com o lote, e o preço pela faixa
q('''
with meses as (select mes, min(dia) as ini, max(dia) as fim from dias where mes between '2026-08' and '2026-10' group by mes),
c as (
  select m.mes,
         count(distinct case when i.anulado = 0 then i.pessoa_id end) as vidas_sem_lote,
         count(distinct i.pessoa_id)                                  as vidas_com_lote
  from meses m join intervalos i
    on i.cliente = 'Delta Benefícios'
   and i.proc_inicio <= m.fim || ' 23:59:59' and (i.proc_fim is null or i.proc_fim >= m.ini)
  group by m.mes
)
select c.mes, n.quantidade as cobrado_vidas, n.valor_unitario as cobrado_preco, n.valor_item as cobrado_valor,
       c.vidas_sem_lote,
       case when c.vidas_sem_lote <= 5000 then 10 when c.vidas_sem_lote <= 10000 then 9 else 8 end as preco_faixa,
       c.vidas_sem_lote * case when c.vidas_sem_lote <= 5000 then 10 when c.vidas_sem_lote <= 10000 then 9 else 8 end as valor_contrato,
       c.vidas_com_lote
from c join notas_emitidas n on n.cliente = 'Delta Benefícios' and n.competencia = c.mes
''')

,mes,cobrado_vidas,cobrado_preco,cobrado_valor,vidas_sem_lote,preco_faixa,valor_contrato,vidas_com_lote
0,2026-08,5051,9.0,45459.0,5270,9,47430,5270
1,2026-09,4941,10.0,49410.0,5300,9,47700,5300
2,2026-10,4892,10.0,48920.0,5105,9,45945,5705


## 7. Faro: pró-rata por dias corridos

Cada vida paga R$ 11,00 × (dias em que esteve cadastrada no mês ÷ dias do mês).

In [13]:
q('''
with m as (select mes, count(*) as dias_do_mes from dias where mes between '2026-08' and '2026-10' group by mes)
select d.mes, m.dias_do_mes,
       count(*) as vida_dias,
       round(11.0 * count(*) / m.dias_do_mes, 2) as valor_pro_rata,
       n.quantidade as cobrado_vidas, n.valor_item as cobrado_valor
from dias d
join m using (mes)
join intervalos i on i.cliente = 'Faro Serviços'
 and i.proc_inicio <= d.dia || ' 23:59:59' and (i.proc_fim is null or i.proc_fim > d.dia || ' 23:59:59')
join notas_emitidas n on n.cliente = 'Faro Serviços' and n.competencia = d.mes
group by d.mes
''')

,mes,dias_do_mes,vida_dias,valor_pro_rata,cobrado_vidas,cobrado_valor
0,2026-08,31,8227,2919.26,266,2926.0
1,2026-09,30,7727,2833.23,258,2838.0
2,2026-10,31,8079,2866.74,261,2871.0


## 8. Cedro: máximo do mês e rateio

O funil: primeiro a quantidade do grupo (máximo de vidas distintas no mês), depois a divisão entre as três empresas.

**Transferido conta uma vez** (apoiado nos dados): a pessoa transferida tem o mesmo `pessoa_id` nos dois CNPJs. De 14 a 19/10, 126 pessoas ficaram cadastradas na Lojas Sul e na Distribuição ao mesmo tempo; contam uma vez, na empresa de destino.

**Rateio pelo dia do pico** (Mateus, 17/11, com ressalva). O contrato (6.3) diz "na proporção das vidas de cada uma" e não diz de qual momento. Nas 9 notas do Cedro, quantidade e divisão são sempre do mesmo instante; por isso a divisão usa o dia em que o máximo foi alcançado. A regra do máximo nunca foi aplicada, então isso é extensão do padrão das notas, não observação direta. Fica anotado como pergunta à Medipreço.

In [14]:
con.executescript('''
drop table if exists cedro_dia;
create temp table cedro_dia as   -- uma linha por dia e pessoa, já sem duplicidade (vale a empresa de entrada mais recente)
select dia, mes, pessoa_id, razao_social
from (select d.dia, d.mes, i.pessoa_id, i.razao_social,
             row_number() over (partition by d.dia, i.pessoa_id order by i.proc_inicio desc) as rn
      from dias d join intervalos i
        on i.cliente = 'Grupo Cedro'
       and i.proc_inicio <= d.dia || ' 23:59:59' and (i.proc_fim is null or i.proc_fim > d.dia || ' 23:59:59'))
where rn = 1;
''')

# Passo 1: a quantidade do grupo
q('''
with g as (select mes, dia, count(*) as vidas from cedro_dia group by mes, dia),
pico as (select mes, max(vidas) as maximo from g group by mes)
select p.mes, p.maximo as maximo_do_mes, min(g.dia) as dia_do_pico, p.maximo * 8 as valor_total,
       (select sum(quantidade) from notas_emitidas n where n.cliente = 'Grupo Cedro' and n.competencia = p.mes) as cobrado_vidas,
       (select sum(valor_item) from notas_emitidas n where n.cliente = 'Grupo Cedro' and n.competencia = p.mes) as cobrado_valor
from pico p join g on g.mes = p.mes and g.vidas = p.maximo
where p.mes between '2026-08' and '2026-10'
group by p.mes
''')

,mes,maximo_do_mes,dia_do_pico,valor_total,cobrado_vidas,cobrado_valor
0,2026-08,996,2026-08-01,7968,968,7744.0
1,2026-09,963,2026-09-01,7704,961,7688.0
2,2026-10,961,2026-10-05,7688,946,7568.0


In [15]:
# Por que o momento importa: a divisão entre as empresas muda dentro de outubro
q('''
select dia,
       sum(razao_social like '%Varejo%')       as varejo,
       sum(razao_social like '%Lojas Sul%')    as lojas_sul,
       sum(razao_social like '%Distribuição%') as distribuicao,
       count(*) as grupo
from cedro_dia
where dia in ('2026-10-02', '2026-10-05', '2026-10-13', '2026-10-14', '2026-10-19', '2026-10-26', '2026-10-30')
group by dia
''')

,dia,varejo,lojas_sul,distribuicao,grupo
0,2026-10-02,427,357,176,960
1,2026-10-05,421,358,182,961
2,2026-10-13,421,355,172,948
3,2026-10-14,421,229,298,948
4,2026-10-19,419,230,297,946
5,2026-10-26,417,228,300,945
6,2026-10-30,417,228,300,945


In [16]:
# Passo 2: o rateio, pelas vidas de cada empresa no dia do pico. Uma nota por empresa; a soma fecha sem centavos.
q('''
with g as (select mes, dia, count(*) as vidas from cedro_dia group by mes, dia),
pico as (select g.mes, min(g.dia) as dia, g.vidas from g
         join (select mes, max(vidas) as mx from g group by mes) m on m.mes = g.mes and m.mx = g.vidas group by g.mes)
select p.mes, p.dia as dia_do_pico, c.razao_social, count(*) as vidas, count(*) * 8 as valor_da_nota,
       (select n.quantidade from notas_emitidas n where n.tomador = c.razao_social and n.competencia = p.mes) as cobrado_vidas,
       (select n.valor_item from notas_emitidas n where n.tomador = c.razao_social and n.competencia = p.mes) as cobrado_valor
from pico p join cedro_dia c on c.dia = p.dia
where p.mes between '2026-08' and '2026-10'
group by p.mes, c.razao_social
''')

,mes,dia_do_pico,razao_social,vidas,valor_da_nota,cobrado_vidas,cobrado_valor
0,2026-08,2026-08-01,Cedro Distribuição Ltda.,178,1424,173,1384.0
1,2026-08,2026-08-01,Cedro Lojas Sul Ltda.,372,2976,364,2912.0
2,2026-08,2026-08-01,Cedro Varejo S.A.,446,3568,431,3448.0
3,2026-09,2026-09-01,Cedro Distribuição Ltda.,173,1384,174,1392.0
4,2026-09,2026-09-01,Cedro Lojas Sul Ltda.,362,2896,357,2856.0
5,2026-09,2026-09-01,Cedro Varejo S.A.,428,3424,430,3440.0
6,2026-10,2026-10-05,Cedro Distribuição Ltda.,182,1456,297,2376.0
7,2026-10,2026-10-05,Cedro Lojas Sul Ltda.,358,2864,230,1840.0
8,2026-10,2026-10-05,Cedro Varejo S.A.,421,3368,419,3352.0


## 9. Estrela: o mínimo de 500 vidas, mês a mês

O contrato cobra R$ 9,50 por vida pela **posição da base no dia 26** (5.1), com **mínimo de 500 vidas** (5.2).

Duas colunas para comparar:
- `base_no_dia_26`: o que estava cadastrado no dia 26 às 9h. É o que o contrato manda cobrar (decisão do Mateus, 17/11) e o que foi cobrado.
- `ativos_reais_no_dia_26`: quem de fato ainda trabalhava lá no dia 26, sabendo o que a Estrela informou depois.

Enquanto a base cadastrada fica acima de 500, o mínimo não entra em cena. Em outubro a base mostrava 510, mas só 479 estavam de fato ativos: a Estrela pagou por 510 porque informou as saídas depois do dia 26.

In [17]:
q('''
with meses as (select '2026-08' as mes union all select '2026-09' union all select '2026-10'),
c as (
  select m.mes,
    (select count(*) from intervalos i where i.cliente = 'Estrela Indústria'
        and i.proc_inicio <= m.mes || '-26 09:00:00' and (i.proc_fim is null or i.proc_fim > m.mes || '-26 09:00:00')) as base_no_dia_26,
    (select count(*) from intervalos i where i.cliente = 'Estrela Indústria'
        and i.inicio <= m.mes || '-26' and (i.fim is null or i.fim >= m.mes || '-26'))                                 as ativos_reais_no_dia_26
  from meses m
)
select c.mes, n.quantidade as cobrado_vidas, n.valor_item as cobrado_valor,
       c.base_no_dia_26,
       c.ativos_reais_no_dia_26,
       500 as minimo,
       case when c.ativos_reais_no_dia_26 < 500 then 'cairia no mínimo' else 'acima do mínimo' end as se_fosse_pelos_ativos_reais,
       max(c.ativos_reais_no_dia_26, 500) * 9.5 as valor_pelos_ativos_reais
from c join notas_emitidas n on n.cliente = 'Estrela Indústria' and n.competencia = c.mes
''')

,mes,cobrado_vidas,cobrado_valor,base_no_dia_26,ativos_reais_no_dia_26,minimo,se_fosse_pelos_ativos_reais,valor_pelos_ativos_reais
0,2026-08,503,4778.5,503,502,500,acima do mínimo,4769.0
1,2026-09,509,4835.5,509,513,500,acima do mínimo,4873.5
2,2026-10,510,4845.0,510,479,500,cairia no mínimo,4750.0


In [18]:
# O arquivo de 05/11: o que ele trouxe sobre outubro, depois que a nota já tinha sido emitida (03/11)
q('''
select tipo_evento,
       case when data_informada <= '2026-10-26' then 'aconteceu até 26/10' else 'aconteceu depois de 26/10' end as quando,
       count(*) as eventos, min(data_informada) as de, max(data_informada) as ate
from ev
where cliente = 'Estrela Indústria' and dia_proc = '2026-11-05'
group by tipo_evento, quando
''')

,tipo_evento,quando,eventos,de,ate
0,entrada,aconteceu até 26/10,7,2026-10-08,2026-10-23
1,entrada,aconteceu depois de 26/10,2,2026-10-29,2026-11-04
2,saida,aconteceu até 26/10,39,2026-10-05,2026-10-26
3,saida,aconteceu depois de 26/10,5,2026-10-27,2026-11-04


In [19]:
# Novembro: a base cadastrada hoje já está abaixo de 500. Se nada mudar até 26/11, o mínimo passa a valer.
q('''
select count(*) as base_cadastrada_hoje, 500 as minimo,
       max(count(*), 500) as vidas_a_cobrar, max(count(*), 500) * 9.5 as valor_novembro,
       500 - count(*) as vidas_de_complemento
from intervalos where cliente = 'Estrela Indústria' and proc_fim is null
''')

,base_cadastrada_hoje,minimo,vidas_a_cobrar,valor_novembro,vidas_de_complemento
0,475,500,500,4750.0,25


## 10. A data da contagem: o contrato diz dia 26, a prática varia

Aurora, Estrela e Bravo têm o dia 26 no contrato. A contagem real (`contagem_em`) foi sempre às 9h, e recuou para a sexta quando o dia 26 caiu em fim de semana. Abaixo, só os casos em que o número mudaria se a contagem fosse feita no fim do dia 26.

Só Aurora, Estrela e Bravo têm essa regra; Faro e Cedro aparecem porque hoje também são cobrados pela foto.

In [20]:
q('''
with inst as (
  select distinct cliente, competencia, contagem_em, base_calculo,
         substr(contagem_em, 1, 8) || '26 23:59:59' as fim_do_dia_26
  from notas_emitidas
),
c as (
  select s.*,
    (select count(*) from intervalos i where i.cliente = s.cliente and (s.base_calculo = 'vidas' or i.tipo = 'titular')
        and i.proc_inicio <= s.contagem_em and (i.proc_fim is null or i.proc_fim > s.contagem_em))     as na_contagem,
    (select count(*) from intervalos i where i.cliente = s.cliente and (s.base_calculo = 'vidas' or i.tipo = 'titular')
        and i.proc_inicio <= s.fim_do_dia_26 and (i.proc_fim is null or i.proc_fim > s.fim_do_dia_26)) as no_fim_do_dia_26
  from inst s
)
select cliente, competencia, contagem_em,
       case strftime('%w', substr(contagem_em, 1, 8) || '26') when '0' then 'domingo' when '6' then 'sábado' else 'dia útil' end as dia_26_caiu_em,
       na_contagem, no_fim_do_dia_26, no_fim_do_dia_26 - na_contagem as diferenca
from c
where na_contagem <> no_fim_do_dia_26
order by contagem_em, cliente
''')

,cliente,competencia,contagem_em,dia_26_caiu_em,na_contagem,no_fim_do_dia_26,diferenca
0,Bravo Tecnologia,2026-09,2026-08-26 09:00:00,dia útil,332,333,1
1,Faro Serviços,2026-08,2026-08-26 09:00:00,dia útil,266,267,1
2,Bravo Tecnologia,2026-10,2026-09-25 09:00:00,sábado,337,336,-1
3,Faro Serviços,2026-09,2026-09-25 09:00:00,sábado,258,257,-1
4,Bravo Tecnologia,2026-11,2026-10-26 09:00:00,dia útil,347,348,1
5,Faro Serviços,2026-10,2026-10-26 09:00:00,dia útil,261,262,1
6,Grupo Cedro,2026-10,2026-10-26 09:00:00,dia útil,946,945,-1


In [21]:
# Quando o sistema processa: cargas só de manhã, em dia útil; não há data de envio do arquivo na base
q('''
select origem,
       min(time(data_processamento)) as hora_mais_cedo, max(time(data_processamento)) as hora_mais_tarde,
       sum(strftime('%w', data_processamento) in ('0', '6')) as eventos_em_fim_de_semana,
       count(*) as eventos
from ev where data_processamento >= '2026-06-01'
group by origem
''')

,origem,hora_mais_cedo,hora_mais_tarde,eventos_em_fim_de_semana,eventos
0,carga_rh,09:00:00,11:57:06,0,3938
1,portal_rh,08:08:58,18:59:09,0,413
2,suporte_medipreco,16:05:00,16:07:29,0,600


In [22]:
# Aurora manda arquivo toda terça. O de 27/10 veio no ritmo normal: não há sinal de fila do lado da Medipreço
q('''
select dia_proc as carga,
       case strftime('%w', dia_proc) when '1' then 'segunda' when '2' then 'terça' when '3' then 'quarta' when '4' then 'quinta' else 'sexta' end as dia_da_semana,
       count(*) as eventos, min(time(data_processamento)) as inicio, max(time(data_processamento)) as fim
from ev where cliente = 'Aurora Logística' and data_processamento >= '2026-09-01'
group by dia_proc
''')

,carga,dia_da_semana,eventos,inicio,fim
0,2026-09-01,terça,5,10:10:00,10:10:01
1,2026-09-08,terça,12,10:27:00,10:27:03
2,2026-09-15,terça,12,11:40:00,11:40:03
3,2026-09-22,terça,8,09:19:00,09:19:02
4,2026-09-29,terça,9,10:46:00,10:46:02
5,2026-10-06,terça,56,11:48:00,11:48:18
6,2026-10-13,terça,10,11:40:00,11:40:03
7,2026-10-20,terça,140,10:33:00,10:33:46
8,2026-10-27,terça,19,11:57:00,11:57:06
9,2026-11-03,terça,18,09:20:00,09:20:05


## 11. Resumo: faturado × regra do contrato

Tudo pela base cadastrada. Regra de cada cliente:
- **Aurora:** vidas por plano na base do dia de contagem. **Estrela:** idem, com mínimo de 500.
- **Bravo:** titulares na base do dia de contagem (cobrança antecipada). O acerto da cláusula 4.4 não é aplicado e não está aqui.
- **Delta:** vidas cadastradas em ao menos 1 dia do mês, sem o lote anulado, pelo preço da faixa.
- **Faro:** pró-rata por dias corridos. **Cedro:** máximo do mês × R$ 8.

In [23]:
resumo = q('''
with meses as (select mes, min(dia) as ini, max(dia) as fim, count(*) as nd from dias where mes between '2026-08' and '2026-10' group by mes),
aurora as (
  select n.competencia as mes, sum(case f.plano when 'Base' then 7.5 else 12.0 end) as valor
  from (select distinct competencia, contagem_em from notas_emitidas where cliente = 'Aurora Logística') n
  join foto f on f.contagem_em = n.contagem_em and f.cliente = 'Aurora Logística' group by 1),
estrela as (
  select n.competencia as mes, max(count(*), 500) * 9.5 as valor
  from (select distinct competencia, contagem_em from notas_emitidas where cliente = 'Estrela Indústria') n
  join foto f on f.contagem_em = n.contagem_em and f.cliente = 'Estrela Indústria' group by 1),
bravo as (
  select n.competencia as mes, sum(f.tipo = 'titular') * 15.0 as valor
  from (select distinct competencia, contagem_em from notas_emitidas where cliente = 'Bravo Tecnologia') n
  join foto f on f.contagem_em = n.contagem_em and f.cliente = 'Bravo Tecnologia' group by 1),
delta as (
  select mes, v * case when v <= 5000 then 10.0 when v <= 10000 then 9.0 else 8.0 end as valor
  from (select m.mes, count(distinct i.pessoa_id) as v from meses m join intervalos i
          on i.cliente = 'Delta Benefícios' and i.anulado = 0
         and i.proc_inicio <= m.fim || ' 23:59:59' and (i.proc_fim is null or i.proc_fim >= m.ini) group by m.mes)),
faro as (
  select d.mes, round(11.0 * count(*) / m.nd, 2) as valor
  from dias d join meses m using (mes) join intervalos i on i.cliente = 'Faro Serviços'
   and i.proc_inicio <= d.dia || ' 23:59:59' and (i.proc_fim is null or i.proc_fim > d.dia || ' 23:59:59') group by d.mes),
cedro as (
  select mes, max(vidas) * 8.0 as valor from (select mes, dia, count(*) as vidas from cedro_dia group by mes, dia) group by mes),
regra as (
  select 'Aurora Logística' as cliente, * from aurora union all select 'Estrela Indústria', * from estrela
  union all select 'Bravo Tecnologia', * from bravo union all select 'Delta Benefícios', * from delta
  union all select 'Faro Serviços', * from faro union all select 'Grupo Cedro', * from cedro),
fat as (select cliente, competencia as mes, sum(valor_item) as faturado from notas_emitidas group by 1, 2)
select f.cliente, f.mes as competencia, f.faturado, r.valor as pela_regra_do_contrato, round(r.valor - f.faturado, 2) as diferenca
from fat f join regra r on r.cliente = f.cliente and r.mes = f.mes
where f.mes between '2026-08' and '2026-10'
order by f.cliente, f.mes
''')
resumo

,cliente,competencia,faturado,pela_regra_do_contrato,diferenca
0,Aurora Logística,2026-08,7359.0,7359.00,0.00
1,Aurora Logística,2026-09,7381.5,7381.50,0.00
2,Aurora Logística,2026-10,6378.0,6378.00,0.00
3,Bravo Tecnologia,2026-08,4815.0,4815.00,0.00
4,Bravo Tecnologia,2026-09,7485.0,4980.00,-2505.00
5,Bravo Tecnologia,2026-10,5055.0,5055.00,0.00
6,Delta Benefícios,2026-08,45459.0,47430.00,1971.00
7,Delta Benefícios,2026-09,49410.0,47700.00,-1710.00
8,Delta Benefícios,2026-10,48920.0,45945.00,-2975.00
9,Estrela Indústria,2026-08,4778.5,4778.50,0.00


In [24]:
# Total por competência
resumo.groupby("competencia")[["faturado", "pela_regra_do_contrato", "diferenca"]].sum().round(2)

,faturado,pela_regra_do_contrato,diferenca
competencia,,,
2026-08,73081.5,75269.76,2188.26
2026-09,79638.0,75434.23,-4203.77
2026-10,75637.0,72777.74,-2859.26
